# Italian Restaurant Problem with AMPL in Python
## AMPL Modeling for Book Problem 2.9


## Requirements

This notebook uses `amplpy` together with the HiGHS solver module.

If your environment does not have `amplpy`, install it first:

```python
%pip install amplpy
```


In [1]:
from __future__ import annotations

from functools import wraps
from time import perf_counter


In [2]:
%pip install amplpy
try:
    from amplpy import ampl_notebook
except ImportError as exc:
    raise ImportError(
        "This notebook requires amplpy. Install it with `%pip install amplpy` before running."
    ) from exc


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper


def create_ampl_instance(solver: str = "highs"):
    return ampl_notebook(modules=[solver], license_uuid="default")


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: /Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
DISHES = ["tallarines", "ravioles", "lasagna", "corbatitas", "fideos"]
INGREDIENTS = ["tomato", "onion", "egg", "pepper"]

PROFIT = {
    "tallarines": 50,
    "ravioles": 65,
    "lasagna": 40,
    "corbatitas": 70,
    "fideos": 55,
}

DEMAND = {
    "tallarines": 15,
    "ravioles": 12,
    "lasagna": 32,
    "corbatitas": 18,
    "fideos": 10,
}

STOCK = {
    "tomato": 80,
    "onion": 75,
    "egg": 10,
    "pepper": 30,
}

USAGE = {
    ("tallarines", "tomato"): 1,
    ("tallarines", "onion"): 1,
    ("tallarines", "egg"): 0,
    ("tallarines", "pepper"): 1,
    ("ravioles", "tomato"): 4,
    ("ravioles", "onion"): 2,
    ("ravioles", "egg"): 0,
    ("ravioles", "pepper"): 1,
    ("lasagna", "tomato"): 2,
    ("lasagna", "onion"): 1,
    ("lasagna", "egg"): 1,
    ("lasagna", "pepper"): 2,
    ("corbatitas", "tomato"): 0,
    ("corbatitas", "onion"): 0,
    ("corbatitas", "egg"): 2,
    ("corbatitas", "pepper"): 0,
    ("fideos", "tomato"): 1,
    ("fideos", "onion"): 1,
    ("fideos", "egg"): 0,
    ("fideos", "pepper"): 1,
}


In [4]:
@timed
def solve_restaurant_with_ampl(minimum_daily_production=None, solver: str = "highs"):
    ampl = create_ampl_instance(solver)
    ampl.eval(
        r'''
        set DISHES ordered;
        set INGREDIENTS ordered;
        param profit {DISHES};
        param demand {DISHES};
        param stock {INGREDIENTS};
        param usage {DISHES, INGREDIENTS};
        param minimum_daily_production default 0;

        var Produce {d in DISHES} integer >= 0;

        maximize TotalProfit:
            sum {d in DISHES} profit[d] * Produce[d];

        subject to IngredientLimit {i in INGREDIENTS}:
            sum {d in DISHES} usage[d, i] * Produce[d] <= stock[i];

        subject to DemandLimit {d in DISHES}:
            Produce[d] <= demand[d];

        subject to MinimumOutput {d in DISHES}:
            Produce[d] >= minimum_daily_production;
        '''
    )
    ampl.set["DISHES"] = DISHES
    ampl.set["INGREDIENTS"] = INGREDIENTS
    ampl.param["profit"] = PROFIT
    ampl.param["demand"] = DEMAND
    ampl.param["stock"] = STOCK
    ampl.param["usage"] = USAGE
    ampl.param["minimum_daily_production"] = 0 if minimum_daily_production is None else minimum_daily_production
    ampl.solve(solver=solver)

    values = ampl.get_variable("Produce").get_values().to_dict()
    return {
        "production": {dish: int(round(values[dish])) for dish in DISHES},
        "profit": int(round(ampl.get_objective("TotalProfit").value())),
    }


In [5]:
base_result, base_time = solve_restaurant_with_ampl()
print("=== BASE RESTAURANT RESULTS WITH AMPL ===")
print(f"Solution -> {base_result}")
print(f"Time     -> {base_time:.8f} seconds")

assert base_result["profit"] == 2080


/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 2080
1 simplex iterations
1 branching nodes
=== BASE RESTAURANT RESULTS WITH AMPL ===
Solution -> {'production': {'tallarines': 8, 'ravioles': 12, 'lasagna': 0, 'corbatitas': 5, 'fideos': 10}, 'profit': 2080}
Time     -> 1.57066213 seconds


In [6]:
variant_result, variant_time = solve_restaurant_with_ampl(minimum_daily_production=3)
print("=== MINIMUM-3-PLATES VARIATION WITH AMPL ===")
print(f"Solution -> {variant_result}")
print(f"Time     -> {variant_time:.8f} seconds")

assert variant_result["profit"] == 1755


AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 1755
1 simplex iterations
1 branching nodes
=== MINIMUM-3-PLATES VARIATION WITH AMPL ===
Solution -> {'production': {'tallarines': 3, 'ravioles': 12, 'lasagna': 3, 'corbatitas': 3, 'fideos': 9}, 'profit': 1755}
Time     -> 1.58703250 seconds
